# Práctica 3: Data Visualization

## Objetivo
Generar visualizaciones claras y precisas para revelar patrones y tendencias en los datos de tiros de la EPL 2024-2025. Se utilizarán bucles (`for` loops) para generar múltiples gráficos de manera eficiente.

## Requisitos
- Generar al menos 5 tipos de diagramas diferentes: Pie, Histogram, Boxplot, Line Plot, Scatter Plot.
- Usar bucles para la generación de gráficos.
- Priorizar la claridad, precisión y diseño efectivo.

## Criterios de Desempeño
1.  **Claridad y Precisión:** Evitar distorsiones, seleccionar el gráfico adecuado.
2.  **Patrones y Tendencias:** Facilitar la identificación de insights.
3.  **Diseño:** Uso efectivo de color y elementos visuales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo para visualizaciones de alta calidad
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Cargar datos
try:
    df = pd.read_csv('cleaned_epl_shots.csv')
    df['match_date'] = pd.to_datetime(df['match_date'])
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("El archivo no se encuentra. Verifica la ruta.")

df.head()

## 1. Histogramas: Distribución de Variables Numéricas
Utilizamos un bucle para generar histogramas de las variables numéricas clave, permitiendo observar la distribución de probabilidad de cada una.

In [ ]:
numeric_cols = ['xg', 'xgot', 'time', 'shot_x', 'shot_y']
titles = {
    'xg': 'Distribución de Expected Goals (xG)',
    'xgot': 'Distribución de xG on Target (xGOT)',
    'time': 'Distribución del Minuto del Tiro',
    'shot_x': 'Distribución de la Coordenada X del Tiro',
    'shot_y': 'Distribución de la Coordenada Y del Tiro'
}

for col in numeric_cols:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[col], kde=True, color='skyblue', bins=30)
    plt.title(titles.get(col, f'Distribución de {col}'))
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

### Interpretación de Histogramas

- **xG (Expected Goals):** La distribución está fuertemente sesgada a la derecha. La gran mayoría de los tiros tienen una probabilidad muy baja de ser gol (xG < 0.10). Esto refleja la naturaleza del fútbol: es difícil generar ocasiones claras de gol.
- **xGOT (xG on Target):** Similar al xG, pero incluye muchos valores en 0 (tiros fuera o bloqueados). Los valores altos indican tiros muy bien ejecutados.
- **Time (Minuto):** La distribución suele ser bastante uniforme, aunque a menudo se observa un ligero aumento hacia el final de cada tiempo (minutos 45+ y 90+) debido a la urgencia de marcar y el tiempo añadido.
- **Coordenadas (Shot X/Y):** Muestran desde dónde se dispara. Esperamos ver picos en posiciones centrales y cercanas a la portería.

## 2. Boxplots: Relación entre Variables Categóricas y Numéricas
Generamos diagramas de caja para analizar cómo varía la calidad del tiro (`xg`) según diferentes categorías (situación, tipo de tiro, posición del jugador).

In [ ]:
categorical_cols = ['situation', 'shotType', 'player_position', 'isHome']
target_metric = 'xg'

for cat_col in categorical_cols:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=cat_col, y=target_metric, data=df, palette='Set2')
    plt.title(f'Distribución de xG por {cat_col}')
    plt.xticks(rotation=45)
    plt.xlabel(cat_col)
    plt.ylabel('Expected Goals (xG)')
    plt.show()

### Interpretación de Boxplots

- **Por Situación:** Es probable que los penales (si están en la data) tengan una mediana de xG mucho más alta (~0.76) y una variabilidad casi nula. Las jugadas a balón parado (córners, tiros libres) suelen tener medianas más bajas que las jugadas abiertas claras.
- **Por Tipo de Tiro:** Los tiros con el pie suelen tener un rango de xG más amplio que los cabezazos, que generalmente son de menor probabilidad a menos que sean muy cerca del arco.
- **Por Posición:** Los delanteros (Forwards) deberían tener una mediana de xG ligeramente superior o más outliers positivos (grandes ocasiones) en comparación con defensores, quienes suelen tirar desde más lejos (xG bajo).

## 3. Scatter Plots: Relación entre Variables Numéricas
Exploramos correlaciones y patrones espaciales. Destacamos el mapa de tiros (`shot_x` vs `shot_y`) y la relación entre `xg` y `xgot`.

In [ ]:
scatter_pairs = [
    ('shot_y', 'shot_x', 'situation'),  # Mapa de tiros (invertimos ejes para simular campo)
    ('xg', 'xgot', 'shotType')          # Calidad vs Ejecución
]

for x_col, y_col, hue_col in scatter_pairs:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df, x=x_col, y=y_col, hue=hue_col, alpha=0.6, palette='viridis')
    
    if x_col == 'shot_y' and y_col == 'shot_x':
        plt.title('Mapa de Tiros (Shot Map)')
        plt.xlabel('Ancho del Campo (Y)')
        plt.ylabel('Largo del Campo (X)')
        plt.gca().invert_yaxis() # Invertir Y para vista desde arriba si es necesario
    else:
        plt.title(f'Relación entre {x_col} y {y_col}')
        
    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

### Interpretación de Scatter Plots

- **Mapa de Tiros (Shot Map):** Visualizamos claramente la "zona de peligro". La mayor densidad de puntos se encuentra dentro del área penal y centralizada. Los tiros desde las bandas o muy lejanos son menos frecuentes.
- **xG vs xGOT:** 
    - Puntos sobre la línea diagonal (xG ≈ xGOT) indican una ejecución promedio.
    - Puntos muy por encima de la diagonal (xGOT > xG) indican una ejecución excelente (tiros a la escuadra, muy potentes).
    - Puntos en el eje X (xGOT = 0) representan tiros bloqueados o desviados, independientemente de la calidad de la ocasión (xG).

## 4. Pie Charts: Composición de Categorías
Visualizamos la proporción de diferentes categorías en el dataset.

In [ ]:
pie_cols = ['shotType', 'situation', 'isHome']

for col in pie_cols:
    data_counts = df[col].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(data_counts, labels=data_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title(f'Composición de Tiros por {col}')
    plt.show()

### Interpretación de Gráficos de Pastel

- **Shot Type:** Predomina el uso del pie derecho (RightFoot), seguido del izquierdo y luego cabezazos. Esto es consistente con la demografía de jugadores diestros.
- **Situation:** La mayoría de los tiros provienen de "Open Play" (juego abierto). Las jugadas a balón parado (SetPiece, Corner) representan una fracción menor pero tácticamente importante.
- **Localía (isHome):** Generalmente se espera una distribución cercana al 50-50, pero a veces los equipos locales tiran ligeramente más debido a la ventaja de campo.

## 5. Line Plots: Tendencias Temporales
Analizamos cómo evoluciona la cantidad de tiros o el xG promedio a lo largo del tiempo del partido.

In [ ]:
# Agrupamos por minuto de juego (time) para ver la tendencia durante un partido promedio
time_metrics = ['xg', 'xgot']

grouped_time = df.groupby('time')[time_metrics].mean().reset_index()

for metric in time_metrics:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=grouped_time, x='time', y=metric, color='purple', linewidth=2.5)
    plt.title(f'Evolución Promedio de {metric} a lo largo del Partido (0-90+ min)')
    plt.xlabel('Minuto de Juego')
    plt.ylabel(f'Promedio {metric}')
    plt.fill_between(grouped_time['time'], grouped_time[metric], alpha=0.2, color='purple')
    plt.show()

### Interpretación de Gráficos de Línea

- **Tendencia Temporal:** Si la línea muestra picos hacia el final (minuto 80-90), indica que, aunque quizás haya menos tiros por cansancio, las ocasiones generadas pueden ser más claras debido a defensas desordenadas o errores.
- **Volatilidad:** Es normal ver mucha volatilidad en los minutos de descuento (45+ y 90+) porque la muestra de datos es menor en esos minutos específicos comparado con los minutos regulares (1-90).

## Conclusiones Generales
A través de estas visualizaciones hemos confirmado patrones clásicos del fútbol:
1.  La dificultad de marcar (xG bajo promedio).
2.  La importancia de la ubicación central para tiros de calidad.
3.  La predominancia del juego abierto sobre el balón parado en volumen de intentos.
4.  La relación directa entre la calidad de la ocasión (xG) y la ejecución (xGOT), donde los mejores jugadores logran superar consistentemente sus expectativas.